# Call the model server
The server runs on your computer (`python app.py`) or online on Render. This notebook sends one image and prints the answer.

In [ ]:
# 1. where is the server? (local first, then your Render URL)
URL = "http://127.0.0.1:7860"
# URL = "https://iaac-team0-model-server.onrender.com"

In [ ]:
# 2. a photo travels as text (base64)
import base64
image_b64 = base64.b64encode(open("examples/sneaker.png", "rb").read()).decode()
print(image_b64[:60], "...")

## The easy way: gradio_client

In [ ]:
# !pip install gradio_client
from gradio_client import Client
client = Client(URL)
result = client.predict(image_b64, api_name="/predict")
{k: v for k, v in result.items() if k != "reconstruction_png_base64"}

## The same call with plain HTTP (what a web app does)
Step 1 starts the job, step 2 reads the answer.

In [ ]:
import requests, json
r = requests.post(f"{URL}/gradio_api/call/predict", json={"data": [image_b64]}, timeout=90)
event_id = r.json()["event_id"]
answer = requests.get(f"{URL}/gradio_api/call/predict/{event_id}", timeout=90).text
data = json.loads([line for line in answer.splitlines() if line.startswith("data:")][-1][5:])[0]
{k: v for k, v in data.items() if k != "reconstruction_png_base64"}

**Normal:** the first call after 15 minutes of silence takes about a minute. The free server is waking up.